In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from datetime import datetime, timezone
import os
from load_dotenv import load_dotenv
load_dotenv()


True

In [2]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("delta-s3-aws") \
    .master("local[*]") \
    .config("spark.jars.packages",
        "io.delta:delta-spark_2.12:3.1.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-1.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.access.key", os.environ['aws_access_key_id']) \
    .config("spark.hadoop.fs.s3a.secret.key", os.environ['aws_secret_access_key']) \
    .config("spark.hadoop.fs.s3a.path.style.access", "false") \
    .config("spark.hadoop.hive.metastore.client.factory.class",
            "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory") \
    .config("spark.sql.warehouse.dir", "s3a://amzn-s3-job-prj/warehouse/") \
    .enableHiveSupport() \
    .getOrCreate()

your 131072x1 screen size is bogus. expect trouble
26/05/08 02:32:04 WARN Utils: Your hostname, kien resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/08 02:32:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/k/job_ete/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ntk04/.ivy2/cache
The jars for the packages stored in: /home/ntk04/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8cc6655a-a23f-4b87-ac0a-221597a25e99;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 390ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-run

In [3]:
import sys
import os
import importlib

# Add pipelines directory to path
pipelines_path = '/mnt/k/job_ete/pipelines'
if pipelines_path not in sys.path:
    sys.path.insert(0, pipelines_path)

# Force reload module
if 'utils.glue_catalog' in sys.modules:
    importlib.reload(sys.modules['utils.glue_catalog'])

# Import Glue Catalog Manager
from utils.glue_catalog import GlueCatalogManager

# Initialize Glue Catalog Manager with AWS credentials
glue_mgr = GlueCatalogManager(
    spark=spark,
    region='us-east-1',
    aws_access_key_id=os.environ.get('aws_access_key_id'),
    aws_secret_access_key=os.environ.get('aws_secret_access_key')
)

In [9]:
# Sync từng table một

glue_mgr.create_database(
    database_name="bronze_db",
    s3_location='s3a://amzn-s3-job-prj/bronze'
)


glue_mgr.create_database(
    database_name="silver_db", 
    s3_location='s3a://amzn-s3-job-prj/silver'
)


glue_mgr.create_database(
    database_name="gold_db",
    s3_location='s3a://amzn-s3-job-prj/gold'
)

Database 'bronze_db' created in Glue Catalog
Database 'silver_db' created in Glue Catalog
Database 'gold_db' created in Glue Catalog


True